In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

EARTH_RADIUS_KM = 6371.0088

In [2]:
viirs = pd.read_csv(
    "viirs_night_2023_2024_eda.csv"
)

viirs["acq_date"] = pd.to_datetime(
    viirs["acq_date"]
)

print("Total VIIRS detections:", len(viirs))

Total VIIRS detections: 323659


In [3]:
TEST_START = "2024-01-01"
TEST_END = "2024-01-31"

test = viirs[
    (viirs["acq_date"] >= TEST_START) &
    (viirs["acq_date"] <= TEST_END)
].copy()

test = test.reset_index(drop=True)

test["detection_id"] = np.arange(len(test))

print("POC VIIRS detections:", len(test))
print(
    "Date range:",
    test["acq_date"].min().date(),
    "to",
    test["acq_date"].max().date()
)

POC VIIRS detections: 13609
Date range: 2024-01-01 to 2024-01-31


In [5]:
event_mapping = pd.read_csv(
    "viirs_poc_detection_event_mapping.csv"
)

event_mapping["acq_date"] = pd.to_datetime(
    event_mapping["acq_date"]
)

print("Mapping rows:", len(event_mapping))
print(
    "Unique candidate events:",
    event_mapping["event_id"].nunique()
)

print(
    "Missing event IDs:",
    event_mapping["event_id"].isna().sum()
)

display(event_mapping.head())

Mapping rows: 13609
Unique candidate events: 3337
Missing event IDs: 0


,acq_date,daily_object_id,event_id,latitude,longitude,frp,bright_ti4,bright_ti5
0,2024-01-01,20240101_0,1,30.5576,79.1191,2.2100,297.1900,277.2400
1,2024-01-01,20240101_0,1,30.5582,79.1202,1.9800,304.9900,277.2700
2,2024-01-01,20240101_1,2,30.0443,80.5822,1.6600,309.7600,277.5900
3,2024-01-01,20240101_2,110,27.3758,95.0869,0.5300,300.6100,282.9400
4,2024-01-01,20240101_3,127,26.7939,94.6880,0.4900,295.2800,284.8700


In [6]:
viirs = pd.read_csv(
    "viirs_night_2023_2024_eda.csv"
)

viirs["acq_date"] = pd.to_datetime(
    viirs["acq_date"]
)

test = viirs[
    (viirs["acq_date"] >= "2024-01-01") &
    (viirs["acq_date"] <= "2024-01-31")
].copy()

test = test.reset_index(drop=True)

print("January VIIRS detections:", len(test))

January VIIRS detections: 13609


In [7]:
mapping_columns = [
    "acq_date",
    "latitude",
    "longitude",
    "frp",
    "bright_ti4",
    "bright_ti5"
]

test = test.merge(
    event_mapping[
        mapping_columns + ["event_id"]
    ],
    on=mapping_columns,
    how="left"
)

print("Mapped detections:", len(test))

print(
    "Missing event IDs:",
    test["event_id"].isna().sum()
)

print(
    "Candidate events:",
    test["event_id"].nunique()
)

Mapped detections: 13609
Missing event IDs: 0
Candidate events: 3337


In [8]:
print("========== EVENT MAPPING CHECK ==========")

print(
    "Expected events : 3337"
)

print(
    "Actual events   :",
    test["event_id"].nunique()
)

print(
    "Expected rows   : 13609"
)

print(
    "Actual rows     :",
    len(test)
)

========== EVENT MAPPING CHECK ==========
Expected events : 3337
Actual events   : 3337
Expected rows   : 13609
Actual rows     : 13609


In [9]:
thermal_features = (
    test
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),

        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
)

thermal_features = thermal_features.fillna(0)

display(thermal_features.head())

,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000


In [10]:
temporal_features = (
    test
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    temporal_features["end_date"]
    - temporal_features["start_date"]
).dt.days + 1

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    / temporal_features["duration_days"]
)

display(temporal_features.head())

,event_id,start_date,end_date,active_days,detection_count,duration_days,activity_frequency
0,1,2024-01-01,2024-01-01,1,2,1,1.0000
1,2,2024-01-01,2024-01-01,1,1,1,1.0000
2,3,2024-01-01,2024-01-29,23,33,29,0.7931
3,4,2024-01-01,2024-01-17,17,39,17,1.0000
4,5,2024-01-01,2024-01-01,1,1,1,1.0000


In [11]:
spatial_features = (
    test
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)

display(spatial_features.head())

,event_id,centroid_lat,centroid_lon
0,1,30.5579,79.1196
1,2,30.0443,80.5822
2,3,27.4695,95.4220
3,4,21.7580,83.8424
4,5,21.7341,83.9805


In [12]:
EARTH_RADIUS_KM = 6371.0088


def haversine_km(
    lat1, lon1,
    lat2, lon2
):

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        2
        * EARTH_RADIUS_KM
        * np.arcsin(np.sqrt(a))
    )

In [13]:
test_spatial = test.merge(
    spatial_features,
    on="event_id",
    how="left",
    suffixes=("", "_event")
)

test_spatial["distance_from_centroid_km"] = (
    haversine_km(
        test_spatial["latitude"],
        test_spatial["longitude"],
        test_spatial["centroid_lat"],
        test_spatial["centroid_lon"]
    )
)

extent_features = (
    test_spatial
    .groupby("event_id")
    ["distance_from_centroid_km"]
    .max()
    .reset_index(
        name="spatial_extent_km"
    )
)

display(extent_features.head())

,event_id,spatial_extent_km
0,1,0.0614
1,2,0.0000
2,3,0.3396
3,4,0.6851
4,5,0.0000


In [14]:
event_features = (
    thermal_features
    .merge(
        temporal_features,
        on="event_id",
        how="left"
    )
    .merge(
        spatial_features,
        on="event_id",
        how="left"
    )
    .merge(
        extent_features,
        on="event_id",
        how="left"
    )
)

print(
    "Event feature table:",
    event_features.shape
)

display(event_features.head())

Event feature table: (3337, 19)


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,centroid_lat,centroid_lon,spatial_extent_km
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212,2024-01-01,2024-01-01,1,2,1,1.0000,30.5579,79.1196,0.0614
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,30.0443,80.5822,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206,2024-01-01,2024-01-29,23,33,29,0.7931,27.4695,95.4220,0.3396
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260,2024-01-01,2024-01-17,17,39,17,1.0000,21.7580,83.8424,0.6851
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,21.7341,83.9805,0.0000


In [15]:
event_features["frp_range"] = (
    event_features["max_frp"]
    - event_features["mean_frp"]
)

event_features["ti4_range"] = (
    event_features["max_bright_ti4"]
    - event_features["mean_bright_ti4"]
)

event_features["ti5_range"] = (
    event_features["max_bright_ti5"]
    - event_features["mean_bright_ti5"]
)

event_features["detections_per_active_day"] = (
    event_features["detection_count"]
    / event_features["active_days"]
)

In [16]:
feature_columns = [
    "event_id",

    # Spatial
    "centroid_lat",
    "centroid_lon",
    "spatial_extent_km",

    # Temporal
    "duration_days",
    "active_days",
    "activity_frequency",

    # Detection behaviour
    "detection_count",
    "detections_per_active_day",

    # FRP
    "mean_frp",
    "max_frp",
    "std_frp",
    "frp_range",

    # Brightness temperature
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "ti4_range",

    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "ti5_range"
]

event_features = event_features[
    feature_columns
].copy()

print(
    "Events:",
    len(event_features)
)

print(
    "Features per event:",
    len(event_features.columns) - 1
)

display(event_features.head())

Events: 3337
Features per event: 20


,event_id,centroid_lat,centroid_lon,spatial_extent_km,duration_days,active_days,activity_frequency,detection_count,detections_per_active_day,mean_frp,max_frp,std_frp,frp_range,mean_bright_ti4,max_bright_ti4,std_bright_ti4,ti4_range,mean_bright_ti5,max_bright_ti5,std_bright_ti5,ti5_range
0,1,30.5579,79.1196,0.0614,1,1,1.0000,2,2.0000,2.0950,2.2100,0.1626,0.1150,301.0900,304.9900,5.5154,3.9000,277.2550,277.2700,0.0212,0.0150
1,2,30.0443,80.5822,0.0000,1,1,1.0000,1,1.0000,1.6600,1.6600,0.0000,0.0000,309.7600,309.7600,0.0000,0.0000,277.5900,277.5900,0.0000,0.0000
2,3,27.4695,95.4220,0.3396,29,23,0.7931,33,1.4348,0.9727,2.0200,0.3573,1.0473,305.5497,316.6600,5.6908,11.1103,282.8742,285.1700,1.9206,2.2958
3,4,21.7580,83.8424,0.6851,17,17,1.0000,39,2.2941,1.9026,4.1900,0.9092,2.2874,311.6979,328.6800,9.1680,16.9821,288.7890,292.2800,2.4260,3.4910
4,5,21.7341,83.9805,0.0000,1,1,1.0000,1,1.0000,1.2100,1.2100,0.0000,0.0000,300.0300,300.0300,0.0000,0.0000,289.4000,289.4000,0.0000,0.0000


In [17]:
print("========== MISSING VALUES ==========")

display(
    event_features
    .isna()
    .sum()
)

========== MISSING VALUES ==========


event_id                     0
centroid_lat                 0
centroid_lon                 0
spatial_extent_km            0
duration_days                0
active_days                  0
activity_frequency           0
detection_count              0
detections_per_active_day    0
mean_frp                     0
max_frp                      0
std_frp                      0
frp_range                    0
mean_bright_ti4              0
max_bright_ti4               0
std_bright_ti4               0
ti4_range                    0
mean_bright_ti5              0
max_bright_ti5               0
std_bright_ti5               0
ti5_range                    0
dtype: int64

In [18]:
display(
    event_features.describe()
)

,event_id,centroid_lat,centroid_lon,spatial_extent_km,duration_days,active_days,activity_frequency,detection_count,detections_per_active_day,mean_frp,max_frp,std_frp,frp_range,mean_bright_ti4,max_bright_ti4,std_bright_ti4,ti4_range,mean_bright_ti5,max_bright_ti5,std_bright_ti5,ti5_range
count,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000
mean,1669.0000,23.2466,78.6755,0.1497,2.3135,1.9865,0.9647,4.0782,1.5422,1.1704,1.4341,0.1891,0.2637,304.2936,307.2598,2.4042,2.9662,283.8385,284.4010,0.5161,0.5624
std,963.4533,6.4104,4.9801,0.2778,4.3424,3.4306,0.1046,17.6588,1.3000,0.8954,1.4775,0.4841,0.8528,7.5809,11.0011,4.3508,6.0640,5.9394,6.0970,1.0625,1.2020
min,1.0000,8.2434,68.5758,0.0000,1.0000,1.0000,0.4286,1.0000,1.0000,0.1100,0.1100,0.0000,-0.0000,295.0100,295.0100,0.0000,0.0000,257.4800,259.2300,0.0000,0.0000
25%,835.0000,17.7903,75.3485,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.6400,0.6700,0.0000,0.0000,299.0167,299.5600,0.0000,0.0000,279.0300,279.3600,0.0000,0.0000
50%,1669.0000,22.3219,77.4900,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9400,1.0400,0.0000,0.0000,302.4833,304.0100,0.0000,0.0000,285.2533,285.7800,0.0000,0.0000
75%,2503.0000,30.1333,80.8587,0.2135,1.0000,1.0000,1.0000,2.0000,2.0000,1.4000,1.6900,0.2121,0.1850,307.0200,311.2600,3.3384,3.1000,288.5455,289.1700,0.6293,0.5367
max,3337.0000,34.6299,97.0755,2.9703,31.0000,31.0000,1.0000,714.0000,26.4444,10.8110,28.6300,9.8776,19.0757,352.4500,367.0000,32.2794,46.9164,297.2300,300.1800,13.7532,10.2815


In [19]:
event_features.to_csv(
    "viirs_poc_event_features.csv",
    index=False
)

print(
    "Saved:",
    len(event_features),
    "candidate thermal events"
)

Saved: 3337 candidate thermal events


In [20]:
event_data = daily_objects.copy()

print("Detections:", len(event_data))
print("Candidate events:", event_data["event_id"].nunique())
print("Missing event IDs:", event_data["event_id"].isna().sum())

NameError: name 'daily_objects' is not defined

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

EARTH_RADIUS_KM = 6371.0088

In [22]:
event_data = pd.read_csv(
    "viirs_poc_detection_event_mapping.csv"
)

event_data["acq_date"] = pd.to_datetime(
    event_data["acq_date"]
)

print("Detections:", len(event_data))
print(
    "Candidate events:",
    event_data["event_id"].nunique()
)
print(
    "Missing event IDs:",
    event_data["event_id"].isna().sum()
)

display(event_data.head())

Detections: 13609
Candidate events: 3337
Missing event IDs: 0


,acq_date,daily_object_id,event_id,latitude,longitude,frp,bright_ti4,bright_ti5
0,2024-01-01,20240101_0,1,30.5576,79.1191,2.2100,297.1900,277.2400
1,2024-01-01,20240101_0,1,30.5582,79.1202,1.9800,304.9900,277.2700
2,2024-01-01,20240101_1,2,30.0443,80.5822,1.6600,309.7600,277.5900
3,2024-01-01,20240101_2,110,27.3758,95.0869,0.5300,300.6100,282.9400
4,2024-01-01,20240101_3,127,26.7939,94.6880,0.4900,295.2800,284.8700


In [23]:
thermal_features = (
    event_data
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),

        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
    .fillna(0)
)

display(thermal_features.head())

,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000


In [24]:
temporal_features = (
    event_data
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    temporal_features["end_date"]
    - temporal_features["start_date"]
).dt.days + 1

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    / temporal_features["duration_days"]
)

display(temporal_features.head())

,event_id,start_date,end_date,active_days,detection_count,duration_days,activity_frequency
0,1,2024-01-01,2024-01-01,1,2,1,1.0000
1,2,2024-01-01,2024-01-01,1,1,1,1.0000
2,3,2024-01-01,2024-01-29,23,33,29,0.7931
3,4,2024-01-01,2024-01-17,17,39,17,1.0000
4,5,2024-01-01,2024-01-01,1,1,1,1.0000


In [25]:
spatial_features = (
    event_data
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)

display(spatial_features.head())

,event_id,centroid_lat,centroid_lon
0,1,30.5579,79.1196
1,2,30.0443,80.5822
2,3,27.4695,95.4220
3,4,21.7580,83.8424
4,5,21.7341,83.9805


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

EARTH_RADIUS_KM = 6371.0088

event_data = pd.read_csv(
    "viirs_poc_detection_event_mapping.csv"
)

event_data["acq_date"] = pd.to_datetime(
    event_data["acq_date"]
)

print("Detections:", len(event_data))
print("Candidate events:", event_data["event_id"].nunique())
print("Missing event IDs:", event_data["event_id"].isna().sum())

Detections: 13609
Candidate events: 3337
Missing event IDs: 0


In [27]:
thermal_features = (
    event_data
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),

        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
    .fillna(0)
)

display(thermal_features.head())

,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000


In [28]:
thermal_features = (
    event_data
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),

        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
    .fillna(0)
)

display(thermal_features.head())

,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000


In [29]:
temporal_features = (
    event_data
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    temporal_features["end_date"]
    - temporal_features["start_date"]
).dt.days + 1

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    / temporal_features["duration_days"]
)

display(temporal_features.head())

,event_id,start_date,end_date,active_days,detection_count,duration_days,activity_frequency
0,1,2024-01-01,2024-01-01,1,2,1,1.0000
1,2,2024-01-01,2024-01-01,1,1,1,1.0000
2,3,2024-01-01,2024-01-29,23,33,29,0.7931
3,4,2024-01-01,2024-01-17,17,39,17,1.0000
4,5,2024-01-01,2024-01-01,1,1,1,1.0000


In [30]:
spatial_features = (
    event_data
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)

display(spatial_features.head())

,event_id,centroid_lat,centroid_lon
0,1,30.5579,79.1196
1,2,30.0443,80.5822
2,3,27.4695,95.4220
3,4,21.7580,83.8424
4,5,21.7341,83.9805


In [31]:
def haversine_km(
    lat1, lon1,
    lat2, lon2
):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        2
        * EARTH_RADIUS_KM
        * np.arcsin(np.sqrt(a))
    )

In [32]:
event_points = event_data.merge(
    spatial_features,
    on="event_id",
    how="left",
    suffixes=("", "_event")
)

event_points["distance_from_centroid_km"] = (
    haversine_km(
        event_points["latitude"],
        event_points["longitude"],
        event_points["centroid_lat"],
        event_points["centroid_lon"]
    )
)

extent_features = (
    event_points
    .groupby("event_id")
    ["distance_from_centroid_km"]
    .max()
    .reset_index(
        name="spatial_extent_km"
    )
)

display(extent_features.head())

,event_id,spatial_extent_km
0,1,0.0614
1,2,0.0000
2,3,0.3396
3,4,0.6851
4,5,0.0000


In [33]:
event_features = (
    thermal_features
    .merge(
        temporal_features,
        on="event_id",
        how="left"
    )
    .merge(
        spatial_features,
        on="event_id",
        how="left"
    )
    .merge(
        extent_features,
        on="event_id",
        how="left"
    )
)

print("Event feature table:", event_features.shape)

display(event_features.head())

Event feature table: (3337, 19)


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,centroid_lat,centroid_lon,spatial_extent_km
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212,2024-01-01,2024-01-01,1,2,1,1.0000,30.5579,79.1196,0.0614
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,30.0443,80.5822,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206,2024-01-01,2024-01-29,23,33,29,0.7931,27.4695,95.4220,0.3396
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260,2024-01-01,2024-01-17,17,39,17,1.0000,21.7580,83.8424,0.6851
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,21.7341,83.9805,0.0000


In [34]:
event_features["frp_range"] = (
    event_features["max_frp"]
    - event_features["mean_frp"]
)

event_features["ti4_range"] = (
    event_features["max_bright_ti4"]
    - event_features["mean_bright_ti4"]
)

event_features["ti5_range"] = (
    event_features["max_bright_ti5"]
    - event_features["mean_bright_ti5"]
)

event_features["detections_per_active_day"] = (
    event_features["detection_count"]
    / event_features["active_days"]
)

In [35]:
feature_columns = [
    "event_id",

    # Spatial
    "centroid_lat",
    "centroid_lon",
    "spatial_extent_km",

    # Temporal
    "start_date",
    "end_date",
    "duration_days",
    "active_days",
    "activity_frequency",

    # Activity
    "detection_count",
    "detections_per_active_day",

    # FRP
    "mean_frp",
    "max_frp",
    "std_frp",
    "frp_range",

    # Brightness
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "ti4_range",

    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "ti5_range"
]

event_features = event_features[
    feature_columns
].copy()

print("Events:", len(event_features))
print("Columns:", len(event_features.columns))

display(event_features.head())

Events: 3337
Columns: 23


,event_id,centroid_lat,centroid_lon,spatial_extent_km,start_date,end_date,duration_days,active_days,activity_frequency,detection_count,detections_per_active_day,mean_frp,max_frp,std_frp,frp_range,mean_bright_ti4,max_bright_ti4,std_bright_ti4,ti4_range,mean_bright_ti5,max_bright_ti5,std_bright_ti5,ti5_range
0,1,30.5579,79.1196,0.0614,2024-01-01,2024-01-01,1,1,1.0000,2,2.0000,2.0950,2.2100,0.1626,0.1150,301.0900,304.9900,5.5154,3.9000,277.2550,277.2700,0.0212,0.0150
1,2,30.0443,80.5822,0.0000,2024-01-01,2024-01-01,1,1,1.0000,1,1.0000,1.6600,1.6600,0.0000,0.0000,309.7600,309.7600,0.0000,0.0000,277.5900,277.5900,0.0000,0.0000
2,3,27.4695,95.4220,0.3396,2024-01-01,2024-01-29,29,23,0.7931,33,1.4348,0.9727,2.0200,0.3573,1.0473,305.5497,316.6600,5.6908,11.1103,282.8742,285.1700,1.9206,2.2958
3,4,21.7580,83.8424,0.6851,2024-01-01,2024-01-17,17,17,1.0000,39,2.2941,1.9026,4.1900,0.9092,2.2874,311.6979,328.6800,9.1680,16.9821,288.7890,292.2800,2.4260,3.4910
4,5,21.7341,83.9805,0.0000,2024-01-01,2024-01-01,1,1,1.0000,1,1.0000,1.2100,1.2100,0.0000,0.0000,300.0300,300.0300,0.0000,0.0000,289.4000,289.4000,0.0000,0.0000


In [36]:
print("========== VIIRS EVENT FEATURE VALIDATION ==========")

print("Events:", event_features["event_id"].nunique())
print("Rows:", len(event_features))

print("\nMissing values:")
display(event_features.isna().sum())

print("\nFeature statistics:")
display(event_features.describe())

========== VIIRS EVENT FEATURE VALIDATION ==========
Events: 3337
Rows: 3337

Missing values:


event_id                     0
centroid_lat                 0
centroid_lon                 0
spatial_extent_km            0
start_date                   0
end_date                     0
duration_days                0
active_days                  0
activity_frequency           0
detection_count              0
detections_per_active_day    0
mean_frp                     0
max_frp                      0
std_frp                      0
frp_range                    0
mean_bright_ti4              0
max_bright_ti4               0
std_bright_ti4               0
ti4_range                    0
mean_bright_ti5              0
max_bright_ti5               0
std_bright_ti5               0
ti5_range                    0
dtype: int64


Feature statistics:


,event_id,centroid_lat,centroid_lon,spatial_extent_km,start_date,end_date,duration_days,active_days,activity_frequency,detection_count,detections_per_active_day,mean_frp,max_frp,std_frp,frp_range,mean_bright_ti4,max_bright_ti4,std_bright_ti4,ti4_range,mean_bright_ti5,max_bright_ti5,std_bright_ti5,ti5_range
count,3337.0000,3337.0000,3337.0000,3337.0000,3337,3337,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000,3337.0000
mean,1669.0000,23.2466,78.6755,0.1497,2024-01-15 20:03:57.338927104,2024-01-17 03:35:19.868145152,2.3135,1.9865,0.9647,4.0782,1.5422,1.1704,1.4341,0.1891,0.2637,304.2936,307.2598,2.4042,2.9662,283.8385,284.4010,0.5161,0.5624
min,1.0000,8.2434,68.5758,0.0000,2024-01-01 00:00:00,2024-01-01 00:00:00,1.0000,1.0000,0.4286,1.0000,1.0000,0.1100,0.1100,0.0000,-0.0000,295.0100,295.0100,0.0000,0.0000,257.4800,259.2300,0.0000,0.0000
25%,835.0000,17.7903,75.3485,0.0000,2024-01-09 00:00:00,2024-01-11 00:00:00,1.0000,1.0000,1.0000,1.0000,1.0000,0.6400,0.6700,0.0000,0.0000,299.0167,299.5600,0.0000,0.0000,279.0300,279.3600,0.0000,0.0000
50%,1669.0000,22.3219,77.4900,0.0000,2024-01-15 00:00:00,2024-01-16 00:00:00,1.0000,1.0000,1.0000,1.0000,1.0000,0.9400,1.0400,0.0000,0.0000,302.4833,304.0100,0.0000,0.0000,285.2533,285.7800,0.0000,0.0000
75%,2503.0000,30.1333,80.8587,0.2135,2024-01-24 00:00:00,2024-01-25 00:00:00,1.0000,1.0000,1.0000,2.0000,2.0000,1.4000,1.6900,0.2121,0.1850,307.0200,311.2600,3.3384,3.1000,288.5455,289.1700,0.6293,0.5367
max,3337.0000,34.6299,97.0755,2.9703,2024-01-31 00:00:00,2024-01-31 00:00:00,31.0000,31.0000,1.0000,714.0000,26.4444,10.8110,28.6300,9.8776,19.0757,352.4500,367.0000,32.2794,46.9164,297.2300,300.1800,13.7532,10.2815
std,963.4533,6.4104,4.9801,0.2778,NaN,NaN,4.3424,3.4306,0.1046,17.6588,1.3000,0.8954,1.4775,0.4841,0.8528,7.5809,11.0011,4.3508,6.0640,5.9394,6.0970,1.0625,1.2020


In [38]:
event_features.to_csv(
    "viirs_poc_event_features.csv",
    index=False
)

print("Saved: viirs_poc_event_features.csv")

Saved: viirs_poc_event_features.csv
